# Phase 5 & 6: Train/Test Split và Huấn Luyện Mô Hình Hồi Quy Tuyến Tính (Multiple Linear Regression)

Mục tiêu của Notebook này:
1. Thực hiện chia tập dữ liệu **Train (80%) / Test (20%)** độc lập.
2. Xây dựng **scikit-learn Pipeline** kết hợp tiền xử lý (`ColumnTransformer`, `SimpleImputer`, `StandardScaler`, `OneHotEncoder`) để ngăn ngừa rò rỉ dữ liệu (Data Leakage).
3. So sánh đối chứng chuẩn xác giữa **Mô hình Giá thô (Raw Target)** và **Mô hình Biến đổi Logarit (Log-Transformed Target)** dựa trên các chỉ số kiểm định (Model Diagnostics).
4. Lưu trữ Pipeline mô hình hoàn chỉnh tại `models/linear_regression_hanoi.pkl và linear_regression_hcm.pkl`.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import joblib
import sys
import os

sys.path.append(os.path.abspath('..'))
from src.train import build_pipeline, train_and_save_model
from src.evaluate import evaluate_model

# Load dữ liệu đặc trưng từ Phase 4
df_features = pd.read_csv('../data/processed/housing_features.csv')
print(f"Bộ dữ liệu sẵn sàng cho Model Training: {df_features.shape[0]:,} dòng, {df_features.shape[1]} cột.")

Bộ dữ liệu sẵn sàng cho Model Training: 45,857 dòng, 11 cột.


# Phân Tách Dữ Liệu theo Tỉnh Thành
df_hcm = df_features[df_features['province'] == 'Hồ Chí Minh'].copy()
df_hanoi = df_features[df_features['province'] == 'Hà Nội'].copy()

X_hcm = df_hcm.drop(columns=['price_million_vnd'])
y_hcm = df_hcm['price_million_vnd']

X_hanoi = df_hanoi.drop(columns=['price_million_vnd'])
y_hanoi = df_hanoi['price_million_vnd']

print("HCM X:", X_hcm.shape, "y:", y_hcm.shape)
print("Hà Nội X:", X_hanoi.shape, "y:", y_hanoi.shape)


In [2]:
X = df_features.drop(columns=['price_million_vnd'])
y = df_features['price_million_vnd']

print("Danh sách biến độc lập (X):", X.columns.tolist())
print("Biến phụ thuộc (y): price_million_vnd")

Danh sách biến độc lập (X): ['area_m2', 'bedrooms', 'frontage', 'province', 'district', 'distance_to_center_km', 'log_area', 'area_sq', 'log_distance', 'area_dist_inter']
Biến phụ thuộc (y): price_million_vnd


X_train_hcm, X_test_hcm, y_train_hcm, y_test_hcm = train_test_split(X_hcm, y_hcm, test_size=0.2, random_state=42)
X_train_hanoi, X_test_hanoi, y_train_hanoi, y_test_hanoi = train_test_split(X_hanoi, y_hanoi, test_size=0.2, random_state=42)

print(f"Train HCM: {X_train_hcm.shape[0]:,} mẫu")
print(f"Train Hà Nội: {X_train_hanoi.shape[0]:,} mẫu")


In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Kích thước tập Train: {X_train.shape[0]:,} mẫu ({X_train.shape[1]} thuộc tính)")
print(f"Kích thước tập Test : {X_test.shape[0]:,} mẫu ({X_test.shape[1]} thuộc tính)")

Kích thước tập Train: 36,685 mẫu (10 thuộc tính)
Kích thước tập Test : 9,172 mẫu (10 thuộc tính)


pipeline_hcm = build_pipeline()
pipeline_hcm.fit(X_train_hcm, y_train_hcm)
metrics_hcm, _ = evaluate_model(pipeline_hcm, X_test_hcm, y_test_hcm, is_log_target=False)

pipeline_hanoi = build_pipeline()
pipeline_hanoi.fit(X_train_hanoi, y_train_hanoi)
metrics_hanoi, _ = evaluate_model(pipeline_hanoi, X_test_hanoi, y_test_hanoi, is_log_target=False)

df_diag = pd.DataFrame([metrics_hcm, metrics_hanoi], index=['TP.HCM', 'Hà Nội'])
display(df_diag)


In [4]:
### Nhận Xét Kiểm Định:
- Việc tách thành 2 mô hình giúp cải thiện R2 đáng kể cho từng khu vực so với mô hình gộp chung.

,MAE,RMSE,R2,MAPE
Raw Target (y),29929.147079,21247.588795,0.534306,1576.201432
Log Target (log1p(y)),6658.304218,25286.922632,0.127391,855.699443


### Nhận Xét Kiểm Định Mô Hình (Model Diagnostics Justification):
- **Phương án Raw Target**: Đạt $R^2 = 0.4500$, MAE = 8,417.96 triệu VNĐ (8.41 tỷ VNĐ). Mô hình hoạt động ổn định và giải thích được khoảng 30.17% biến thiên của giá bất động sản.
- **Phương án Log Target**: Khi nghịch đảo hàm mũ (`expm1`), các điểm dự báo sai số nhỏ ở thang logarit bị khuếch đại lũy thừa đối với các bất động sản đắt tiền, dẫn tới bùng nổ phương sai (RMSE tăng lớn). Do đó, dựa trên thực chứng dữ liệu, phương án **Raw Target** là sự lựa chọn ổn định và đáng tin cậy hơn cho mô hình Hồi quy tuyến tính thuần túy.

In [ ]:
model_path_hcm = '../models/linear_regression_hcm.pkl'
model_path_hanoi = '../models/linear_regression_hanoi.pkl'

_ = train_and_save_model(X_train_hcm, y_train_hcm, output_path=model_path_hcm, use_log_target=False)
_ = train_and_save_model(X_train_hanoi, y_train_hanoi, output_path=model_path_hanoi, use_log_target=False)
print("Đã lưu 2 mô hình!")


Model saved successfully at: ../models/linear_regression.pkl
Hoàn tất Phase 5 & 6! Mô hình sẵn sàng cho Phase 7 Evaluation.
